In [71]:
import xarray as xr
import matplotlib.pyplot as plt
import imageio.v2 as imageio 
import numpy as np
import os
import sys
import glob 
import zarr 
import torch.nn as nn
import torch
import torch.nn.functional as F
import torch.optim as optim
import torch.cuda.amp as amp

In [3]:
DATA_BASE_PATH = '/Users/navneetmelarkode/Documents/Code/projects/IceConc/data'
TARGET_YEAR = '2020'
PREDICTION_FILE_PATTERN = os.path.join(DATA_BASE_PATH, f"*-amsr_{TARGET_YEAR}*.nc")
ACTUAL_FILE_PATTERN = os.path.join(DATA_BASE_PATH, f"*-amsr_{TARGET_YEAR}*.nc")

In [12]:
actual_files = glob.glob(ACTUAL_FILE_PATTERN)

In [13]:
actual_values_raw = xr.open_mfdataset(
    actual_files,
    combine='by_coords',
    chunks={'time':'auto','yc':'auto','xc':'auto'},
    decode_times=True
)['ice_conc']

In [14]:
actual_values = (actual_values_raw.fillna(0.0).astype('float32') / 100.0)

In [66]:
# Core Hyperparameters
BATCH_SIZE = 2
IMAGE_SIZE = (432, 432)
LEARNING_RATE = 0.001
SEQUENCE_LENGTH = 3
EPOCHS = 20 # Increased epochs since we are sampling less data per epoch
NUM_LAYERS = 3
INPUT_CHANNELS = 1
HIDDEN_CHANNELS = [32, 64, 128] # Reduced for memory efficiency
KERNEL_SIZE = (3, 3)

LOG_INTERVAL = 5            # Log training loss every 5 batches
VAL_INTERVAL = 1            # Run validation every epoch
TRAIN_SAMPLE_PERCENT = 0.25 

In [67]:
import torch
import torch.nn as nn

class ConvLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size, kernel_size, bias):
        """
        input_size: channels of input tensor
        hidden_size: channels of hidden state
        kernel_size: size of the convolutional kernel
        bias: whether to use bias in the convolutional layer
        """
        super(ConvLSTMCell, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.kernel_size = kernel_size
        self.bias = bias
        self.padding = kernel_size[0] // 2, kernel_size[1] // 2

        self.conv = nn.Conv2d(in_channels=self.input_size + self.hidden_size,
                              out_channels=4 * self.hidden_size,
                              kernel_size=self.kernel_size,
                              padding=self.padding,
                              bias=self.bias)

        self.Wci = nn.Conv2d(in_channels=self.hidden_size, out_channels=self.hidden_size, kernel_size=1, bias=self.bias)
        self.Wcf = nn.Conv2d(in_channels=self.hidden_size, out_channels=self.hidden_size, kernel_size=1, bias=self.bias)
        self.Wco = nn.Conv2d(in_channels=self.hidden_size, out_channels=self.hidden_size, kernel_size=1, bias=self.bias)


    def forward(self, x, curr_state):
        h_cur, c_cur = curr_state
        combined = torch.cat([x, h_cur], dim=1)
        combined_conv = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv,
                                             split_size_or_sections=self.hidden_size,
                                             dim=1)
        i = torch.sigmoid(cc_i + self.Wci(c_cur))
        f = torch.sigmoid(cc_f + self.Wcf(c_cur))
        g = torch.tanh(cc_g)

        c_next = f * c_cur + i * g

        o = torch.sigmoid(cc_o + self.Wco(c_next))

        h_next = o * torch.tanh(c_next)

        return h_next, c_next

class ConvLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, kernel_size, num_layers, bias=True, return_sequences=False):
        super(ConvLSTM, self).__init__()
        self.input_size = input_size
        self.hidden_size = self._multilayer_support(hidden_size, num_layers)
        self.kernel_size = self._multilayer_support(kernel_size, num_layers)
        self.num_layers = num_layers
        self.bias = bias
        self.return_sequences = return_sequences
        cell_list = []

        for i in range(0, self.num_layers):
            curr_input_size = self.input_size if i == 0 else self.hidden_size[i-1]
            cell_list.append(ConvLSTMCell(input_size=curr_input_size,
                                          hidden_size=self.hidden_size[i],
                                          kernel_size=self.kernel_size[i],
                                          bias=self.bias))

        self.cell_list = nn.ModuleList(cell_list)

        self.output_projection = nn.Conv2d(in_channels=self.hidden_size[-1],
                                           out_channels=self.input_size,
                                           kernel_size=1,
                                           bias=self.bias)

        self.output_activation = nn.Sigmoid()

    def forward(self, x, hidden_state=None):
        b, seq_len, _, h, w = x.size()

        if hidden_state is None:
            hidden_state = self._init_hidden(b, (h, w), x.device)
        else:
            pass

        last_layer_output_sequence = []

        for t in range(seq_len):
            x_t = x[:,t,:,:,:]

            next_hidden_t = []

            #unrolling

            for idx in range(self.num_layers):
                cell = self.cell_list[idx]

                h_cur, c_cur = hidden_state[idx]
                h_next, c_next = cell(x=x_t, curr_state=[h_cur, c_cur])

                next_hidden_t.append((h_next, c_next))

                x_t = h_next

            hidden_state = next_hidden_t
            last_layer_output_sequence.append(x_t)

        last_layer_output_sequence = torch.stack(last_layer_output_sequence, dim = 0)
        #now seq_len is the first dimension
        last_layer_output_sequence = last_layer_output_sequence.permute(1, 0, 2, 3, 4)

        if self.return_sequences:
            projected_output = self.output_projection(last_layer_output_sequence.view(-1, self.hidden_size[-1], h, w))
            projected_output = projected_output.view(b, seq_len, self.input_size, h, w)
        else:
            projected_output = self.output_projection(last_layer_output_sequence[:,-1,:,:,:])


        final_prediction = self.output_activation(projected_output)

        return final_prediction


    def _init_hidden(self, batch_size, image_size, device):
        H, W = image_size
        init_states = []
        for i in range(self.num_layers):
            init_states.append((torch.zeros(batch_size, self.hidden_size[i], H, W, device=device),
                                torch.zeros(batch_size, self.hidden_size[i], H, W, device=device)))
        return init_states

    @staticmethod
    def _multilayer_support(x, num_layers):
        if (not isinstance(x, list)):
            x = [x] * num_layers
        return x


In [73]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ConvLSTM(
    input_size=INPUT_CHANNELS,
    hidden_size=HIDDEN_CHANNELS,
    kernel_size=KERNEL_SIZE,
    num_layers=NUM_LAYERS
).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = amp.GradScaler()

/var/folders/xh/vh45__8d5rxb0s24ffrjh3nw0000gn/T/ipykernel_7085/3633591817.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()
/Users/navneetmelarkode/.pyenv/versions/cds-env/lib/python3.10/site-packages/torch/amp/grad_scaler.py:136: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


In [74]:
MODEL_CHECKPOINT_PATH = '../trained_model/convlstm_best_model.pth'

checkpoint = torch.load(MODEL_CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scaler.load_state_dict(checkpoint['scaler_state_dict'])
start_epoch = checkpoint['epoch']
best_val_loss = checkpoint['best_val_loss']
global_step = checkpoint['global_step']

In [76]:
data_np = actual_values.values # (time, y, x)
time_coords = actual_values['time'].values
height, width = data_np.shape[1], data_np.shape[2]

In [77]:
height, width

(432, 432)

In [96]:
predicted_data_np = np.full_like(data_np, np.nan, dtype=np.float32)

In [97]:
with torch.no_grad():
    # Initialize the current input sequence with the first SEQUENCE_LENGTH frames from actual data.
    # This sequence will "roll" as new predictions are made.
    current_input_sequence = data_np[0:SEQUENCE_LENGTH] # Ensure SEQUENCE_LENGTH is defined (e.g., 10)

    # Fill the initial part of predicted_data_np with the actual data.
    # No predictions can be made for these initial steps, as they serve as the input history.
    predicted_data_np[:SEQUENCE_LENGTH, :, :] = data_np[:SEQUENCE_LENGTH, :, :]

    # Loop to generate predictions.
    # The loop starts from `SEQ_LEN`, because `data_np[SEQ_LEN]` is the first timestep
    # for which we can make a prediction using `data_np[0:SEQUENCE_LENGTH]` as input.
    for t in range(SEQUENCE_LENGTH, data_np.shape[0]):
        # Convert the current input sequence to tensor: (batch, seq_len, channels, H, W)
        # Unsqueeze(2) adds the channel dimension (1 for ice concentration).
        input_tensor = torch.from_numpy(current_input_sequence).float().unsqueeze(0).unsqueeze(2).to(device)

        model.eval() # Set the model to evaluation mode

        # Make prediction. If the model only returns one tensor, this is the correct way to unpack.
        # The output_sequence is expected to be (batch, seq_len, H, W) after channel squeeze.
        output_sequence = model(input_tensor)

        # Extract the predicted frame.
        # This takes the last frame of the output sequence: output_sequence[batch_idx, last_timestep, H, W]
        predicted_frame = output_sequence[0, -1, :, :].cpu().numpy() # Remove batch, sequence dimensions; no channel index needed

        # Store the predicted frame in the `predicted_data_np` array at the current time step `t`.
        predicted_data_np[t, :, :] = predicted_frame

        # Prepare input for the next iteration (rolling window).
        # Shift the current_input_sequence by removing the oldest frame (first element)
        # and adding the newly predicted frame at the end.
        current_input_sequence = np.roll(current_input_sequence, shift=-1, axis=0)
        current_input_sequence[-1] = predicted_frame

In [98]:
predicted_data_np

array([[[0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        ...,
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ]],

       [[0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        ...,
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0. 

In [99]:
predictions_xr = xr.DataArray(
    predicted_data_np,
    coords={
        'time': actual_values['time'].values,
        'yc': actual_values['yc'].values,
        'xc': actual_values['xc'].values
    },
    dims=['time', 'yc', 'xc'],
    name='ice_conc_predicted'
)

In [100]:
actual_values_for_comparison = actual_values.isel(time=slice(SEQUENCE_LENGTH -1, None))
predictions_for_comparison = predictions_xr.isel(time=slice(SEQUENCE_LENGTH -1, None))

In [101]:
def create_ice_conc_gif(data_array, time_coords, title, output_filename):
    if data_array is None or data_array.isnull().all(): # Check for all NaNs too
        print(f"Cannot create GIF for {title}: data is not loaded or all NaN.")
        return None

    print(f"\nCreating GIF for {title}...")
    output_dir = "visualizations/"
    os.makedirs(output_dir, exist_ok=True)

    frames = []
    # Ensure data_array is loaded to memory for iteration
    data_array_loaded = data_array.load()

    for i, time_step in enumerate(time_coords):
        # Select the frame based on its index
        current_frame_data = data_array_loaded.isel(time=i).values

        # Skip if the frame is all NaNs (e.g., for early unpredicted frames)
        if np.all(np.isnan(current_frame_data)):
            continue

        # Convert to NumPy float32, crucial for image libraries and consistency
        current_frame_data = np.asarray(current_frame_data, dtype=np.float32)

        fig, ax = plt.subplots(figsize=(8, 8))
        img = ax.imshow(current_frame_data, cmap='Blues_r', vmin=0, vmax=1)
        ax.set_title(f'{title}\nTime: {np.datetime_as_string(time_step, unit="D")}')
        ax.axis('off')
        plt.colorbar(img, ax=ax, label='Ice Concentration (0-1)')

        frame_path = os.path.join(output_dir, f'frame_{title.replace(" ", "_")}_{i:04d}.png')
        plt.savefig(frame_path, bbox_inches='tight')
        plt.close(fig)

        try:
            read_frame = imageio.imread(frame_path)
            frames.append(read_frame)
        except Exception as read_err:
            print(f"  ERROR: Could not read frame {i} with imageio: {read_err}")
            print(f"  Skipping frame {i} for GIF creation. Check {frame_path} manually.")
        os.remove(frame_path)

    if frames:
        gif_path = os.path.join(output_dir, output_filename)
        try:
            imageio.mimsave(gif_path, frames, fps=10)
            print(f"GIF saved to {gif_path}")
            return gif_path
        except Exception as gif_err:
            print(f"ERROR: Could not save GIF '{output_filename}': {gif_err}")
            return None
    else:
        print(f"No valid frames were successfully processed for {output_filename}. GIF not created.")
        return None

In [102]:
pred_gif_path = create_ice_conc_gif(predictions_for_comparison, predictions_for_comparison['time'].values,
                                    "Predicted Ice Concentration (Model Output)", "predicted_ice_conc_model_output.gif")
actual_gif_path = create_ice_conc_gif(actual_values_for_comparison, actual_values_for_comparison['time'].values,
                                      "Actual Ice Concentration (Ground Truth)", "actual_ice_conc_ground_truth.gif")



Creating GIF for Predicted Ice Concentration (Model Output)...
GIF saved to visualizations/predicted_ice_conc_model_output.gif

Creating GIF for Actual Ice Concentration (Ground Truth)...
GIF saved to visualizations/actual_ice_conc_ground_truth.gif


In [103]:
def create_comparison_gif(pred_data_xr, actual_data_xr, output_filename):
    if pred_data_xr is None or actual_data_xr is None or pred_data_xr.isnull().all() or actual_data_xr.isnull().all():
        print("Cannot create comparison GIF: data not loaded or all NaN.")
        return None

    frames = []
    output_dir = "visualizations/"
    os.makedirs(output_dir, exist_ok=True)

    # Ensure data is loaded into memory
    pred_data_loaded = pred_data_xr.load()
    actual_data_loaded = actual_data_xr.load()

    # Determine common time steps for a clean comparison
    common_times = np.intersect1d(pred_data_loaded['time'].values, actual_data_loaded['time'].values)
    if len(common_times) == 0:
        print("No common time steps for comparison GIF.")
        return None

    pred_data_common = pred_data_loaded.sel(time=common_times)
    actual_data_common = actual_data_loaded.sel(time=common_times)

    for i, time_step in enumerate(common_times):
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        pred_frame_data = np.asarray(pred_data_common.isel(time=i).values, dtype=np.float32)
        actual_frame_data = np.asarray(actual_data_common.isel(time=i).values, dtype=np.float32)

        # Predicted
        im_pred = axes[0].imshow(pred_frame_data, cmap='Blues_r', vmin=0, vmax=1)
        axes[0].set_title(f'Predicted Ice Conc.\nTime: {np.datetime_as_string(time_step, unit="D")}')
        axes[0].axis('off')
        plt.colorbar(im_pred, ax=axes[0], label='Ice Concentration (0-1)', fraction=0.046, pad=0.04)

        # Actual
        im_actual = axes[1].imshow(actual_frame_data, cmap='Blues_r', vmin=0, vmax=1)
        axes[1].set_title(f'Actual Ice Conc. (Ground Truth)\nTime: {np.datetime_as_string(time_step, unit="D")}')
        axes[1].axis('off')
        plt.colorbar(im_actual, ax=axes[1], label='Ice Concentration (0-1)', fraction=0.046, pad=0.04)

        frame_path = os.path.join(output_dir, f'comp_frame_{i:04d}.png')
        plt.savefig(frame_path, bbox_inches='tight')
        plt.close(fig)
        try:
            read_frame = imageio.imread(frame_path)
            frames.append(read_frame)
        except Exception as read_err:
            print(f"  ERROR (Comparison GIF): Could not read frame {i} with imageio: {read_err}")
            print(f"  Skipping frame {i} for comparison GIF. Check {frame_path} manually.")
        os.remove(frame_path)

    if frames:
        gif_path = os.path.join(output_dir, output_filename)
        try:
            imageio.mimsave(gif_path, frames, fps=10)
            print(f"Comparison GIF saved to {gif_path}")
            return gif_path
        except Exception as gif_err:
            print(f"ERROR (Comparison GIF): Could not save GIF '{output_filename}': {gif_err}")
            return None
    else:
        print(f"No frames were successfully processed for {output_filename}. Comparison GIF not created.")
        return None

comp_gif_path = create_comparison_gif(predictions_for_comparison, actual_values_for_comparison, "comparison_ice_conc_model_vs_actual.gif")


Comparison GIF saved to visualizations/comparison_ice_conc_model_vs_actual.gif
